In [ ]:
# ============================================================
# Cell 0: Auto-discover Kaggle mounted paths
# ============================================================
import os

def find_dir_with_keywords(keywords, base_dir="/kaggle/input"):
    """Recursively find a directory whose name contains all given keywords."""
    for root, dirs, files in os.walk(base_dir):
        for d in dirs:
            if all(k.lower() in d.lower() for k in keywords):
                return os.path.join(root, d)
    return None

def find_model_dir(top_dir):
    """Find the actual sub-directory that contains model weights.

    A valid model directory must contain both config.json and either
    model.safetensors or pytorch_model.bin.
    """
    if not top_dir or not os.path.exists(top_dir):
        return None
    candidates = []
    for root, dirs, files in os.walk(top_dir):
        has_config  = "config.json" in files
        has_weights = any(f in files for f in ("model.safetensors", "pytorch_model.bin"))
        if has_config and has_weights:
            candidates.append(root)
    # Prefer the shortest path among candidates (typically the canonical one)
    return min(candidates, key=len) if candidates else None

def find_lora_dir(top_dir):
    """A valid LoRA directory contains adapter_config.json."""
    if not top_dir or not os.path.exists(top_dir):
        return None
    candidates = []
    for root, dirs, files in os.walk(top_dir):
        if "adapter_config.json" in files:
            candidates.append(root)
    return min(candidates, key=len) if candidates else None

print("Scanning Kaggle mounted paths...\n")

MODEL_TOP = find_dir_with_keywords(["SmolVLM", "500M", "Instruct"])
MODEL_PATH = find_model_dir(MODEL_TOP)

LORA_TOP = find_dir_with_keywords(["lora"])
LORA_PATH = find_lora_dir(LORA_TOP)

print(f"Model top directory:    {MODEL_TOP}")
print(f"Model weight directory: {MODEL_PATH}")
print(f"LoRA top directory:     {LORA_TOP}")
print(f"LoRA weight directory:  {LORA_PATH}")

paths_ok = True
if MODEL_PATH:
    files = sorted(os.listdir(MODEL_PATH))
    print(f"\nMODEL_PATH contents: {files[:10]}")
else:
    print("\nNo model weight directory found.")
    paths_ok = False

if LORA_PATH:
    files = sorted(os.listdir(LORA_PATH))
    print(f"LORA_PATH  contents: {files[:10]}")
else:
    print("No LoRA directory containing adapter_config.json found.")
    paths_ok = False

if not paths_ok:
    raise RuntimeError("Path validation failed.")

import transformers, peft
print(f"\ntransformers: {transformers.__version__}")
print(f"peft:         {peft.__version__}")
print("\nPaths locked in; proceed to Cell 1.")

In [ ]:
# ============================================================
# Cell 1: Environment check (transformers 5.0 compatible)
# ============================================================
# transformers 5.0 removed AutoModelForVision2Seq and renamed it to
# AutoModelForImageTextToText. Kaggle's default image is already on 5.0,
# so no offline wheel install is needed.
try:
    from transformers import AutoProcessor, AutoModelForImageTextToText
    print("AutoModelForImageTextToText imported (transformers 5.0+).")
except ImportError:
    # Fallback for transformers 4.x environments
    from transformers import AutoProcessor, AutoModelForVision2Seq as AutoModelForImageTextToText
    print("Fell back to AutoModelForVision2Seq (transformers 4.x).")

print("Environment check passed; proceed to Cell 2.")

In [ ]:
# ============================================================
# Cell 2: SmolVLM-500M LoRA Inference Pipeline
# ============================================================
import os
import ast
import json
import re
import random
import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm

try:
    from transformers import AutoProcessor, AutoModelForImageTextToText
except ImportError:
    from transformers import AutoProcessor, AutoModelForVision2Seq as AutoModelForImageTextToText

from peft import PeftModel

# ------------------------------------------------------------------
# 1. Reproducibility
# ------------------------------------------------------------------
def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
seed_everything(42)

# ------------------------------------------------------------------
# 2. Paths (offline)
# MODEL_PATH and LORA_PATH are resolved by Cell 0
# ------------------------------------------------------------------
if os.path.exists("/kaggle/input/pixels-to-predictions/test.csv"):
    BASE_COMP_DIR = "/kaggle/input/pixels-to-predictions"
elif os.path.exists("/kaggle/input/competitions/pixels-to-predictions/test.csv"):
    BASE_COMP_DIR = "/kaggle/input/competitions/pixels-to-predictions"
else:
    raise FileNotFoundError("Competition dataset not found.")

TEST_CSV = os.path.join(BASE_COMP_DIR, "test.csv")
candidate_img_dirs = [
    os.path.join(BASE_COMP_DIR, "images/test"),
    os.path.join(BASE_COMP_DIR, "images/images/test"),
    os.path.join(BASE_COMP_DIR, "test"),
]
IMAGE_DIR = next((d for d in candidate_img_dirs if os.path.isdir(d)), None)
assert IMAGE_DIR, f"Test image directory not found: {candidate_img_dirs}"

print(f"Test CSV:    {TEST_CSV}")
print(f"Test images: {IMAGE_DIR}")
print(f"Base model:  {MODEL_PATH}")
print(f"LoRA:        {LORA_PATH}")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.float16 if torch.cuda.is_available() else torch.float32
print(f"Device: {DEVICE}, dtype: {DTYPE}")

# ------------------------------------------------------------------
# 3. Load base model + LoRA + processor
# transformers 5.0 uses dtype= instead of torch_dtype=
# ------------------------------------------------------------------
print("\nLoading SmolVLM-500M base...")
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_PATH,
    dtype=DTYPE,
    device_map="auto",
    low_cpu_mem_usage=True,
)

print("Mounting LoRA...")
model = PeftModel.from_pretrained(model, LORA_PATH)
model.eval()

print("Loading processor...")
# Prefer the processor saved alongside the LoRA weights (matches training).
# Fall back to the base model's processor if the LoRA dir does not have one.
PROCESSOR_PATH = LORA_PATH if os.path.exists(
    os.path.join(LORA_PATH, "processor_config.json")
) else MODEL_PATH
processor = AutoProcessor.from_pretrained(PROCESSOR_PATH)
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token
print(f"Processor loaded from: {PROCESSOR_PATH}")

# ------------------------------------------------------------------
# 4. Prompt builder (must match training exactly)
# ------------------------------------------------------------------
CHOICE_LETTERS    = "ABCDEFGHIJ"
MAX_LECTURE_CHARS = 1200
MAX_HINT_CHARS    = 600

def parse_choices(raw):
    if isinstance(raw, list): return raw
    s = str(raw)
    try: return json.loads(s)
    except Exception:
        try: return ast.literal_eval(s)
        except Exception: return []

def pad_to_square(img, fill_color=(255, 255, 255)):
    w, h = img.size
    if w == h: return img
    side = max(w, h)
    new_img = Image.new("RGB", (side, side), fill_color)
    new_img.paste(img, ((side - w) // 2, (side - h) // 2))
    return new_img

def build_user_prompt(row):
    context_parts = []
    if 'lecture' in row and pd.notna(row['lecture']) and str(row['lecture']).strip():
        lec = str(row['lecture']).strip()
        if len(lec) > MAX_LECTURE_CHARS:
            half = MAX_LECTURE_CHARS // 2
            lec = lec[:half] + " ... " + lec[-half:]
        context_parts.append(lec)
    if 'hint' in row and pd.notna(row['hint']) and str(row['hint']).strip():
        hint = str(row['hint']).strip()
        if len(hint) > MAX_HINT_CHARS:
            hint = hint[:MAX_HINT_CHARS] + " ..."
        context_parts.append(hint)

    context_str = "\n".join(context_parts)
    choices     = parse_choices(row['choices'])
    choices_str = "\n".join(f"  {CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices))

    prompt = ""
    if context_str:
        prompt += f"Context:\n{context_str}\n\n"
    prompt += f"Question: {row['question']}\n"
    prompt += f"Choices:\n{choices_str}\n"
    last_letter = CHOICE_LETTERS[len(choices) - 1] if choices else 'A'
    prompt += f"Pick ONE letter from A to {last_letter}.\nAnswer:"
    return prompt, choices

def parse_letter_to_idx(text: str, n_choices: int) -> int:
    if not text:
        return 0
    valid = set(CHOICE_LETTERS[:n_choices])
    for ch in text.upper():
        if ch in valid:
            return CHOICE_LETTERS.index(ch)
    m = re.search(r'\d+', text)
    if m:
        v = int(m.group())
        return v if 0 <= v < n_choices else 0
    return 0

# ------------------------------------------------------------------
# 5. Inference loop (logit-based scoring)
# ------------------------------------------------------------------
# A single forward pass returns logits over the full vocabulary at the last
# position. We restrict attention to the candidate letter token ids and pick
# the argmax. This avoids the cost and parsing risk of autoregressive
# generation, and it can never emit an illegal answer.
import torch.nn.functional as F

# Pre-compute the token id for each answer letter (A..J) prefixed by a space,
# matching how the assistant response is rendered during training.
LETTER_TOKEN_IDS = []
for letter in CHOICE_LETTERS:
    ids = processor.tokenizer.encode(f" {letter}", add_special_tokens=False)
    LETTER_TOKEN_IDS.append(ids[-1])

LETTER_TOKEN_IDS = torch.tensor(LETTER_TOKEN_IDS, device=DEVICE)
print(f"Letter token ids (first 5): {LETTER_TOKEN_IDS[:5].tolist()}")

# Verification: decoding each id back should yield the original letter
for i, lid in enumerate(LETTER_TOKEN_IDS[:5].tolist()):
    decoded = processor.tokenizer.decode([lid])
    print(f"   {CHOICE_LETTERS[i]} -> token_id={lid} -> decoded={decoded!r}")

test_df = pd.read_csv(TEST_CSV)
print(f"\nRunning logit-based inference on {len(test_df)} samples ...\n")

results = []
err_count = 0

for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
    try:
        img_name = str(row['image_path']).replace('\\', '/').split('/')[-1]
        img_path = os.path.join(IMAGE_DIR, img_name)
        image = Image.open(img_path).convert("RGB")
        image = pad_to_square(image)

        user_text, choices = build_user_prompt(row)
        n_choices = len(choices) if choices else 4

        messages = [{
            "role": "user",
            "content": [{"type": "image"}, {"type": "text", "text": user_text}],
        }]
        prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
        inputs = processor(
            text=[prompt], images=[[image]], return_tensors="pt"
        ).to(DEVICE)

        # Single forward pass; read logits at the last position.
        with torch.no_grad():
            outputs = model(**inputs)

        next_token_logits = outputs.logits[0, -1, :]
        candidate_token_ids = LETTER_TOKEN_IDS[:n_choices]
        candidate_logits = next_token_logits[candidate_token_ids]
        pred = int(candidate_logits.argmax().item())

        results.append({"id": row['id'], "answer": pred})

    except Exception as e:
        err_count += 1
        if err_count <= 5:
            print(f"Error on id={row.get('id','?')}: {e}")
        results.append({"id": row.get('id', 'Unknown'), "answer": 0})

if err_count > 0:
    print(f"\n{err_count} samples failed; default answer 0 used as fallback.")

# ------------------------------------------------------------------
# 6. Submission
# ------------------------------------------------------------------
sub = pd.DataFrame(results)[["id", "answer"]]
sub.to_csv("submission.csv", index=False)

print("\n" + "=" * 60)
print("submission.csv generated.")
print("=" * 60)
print(sub.head())
print(f"\nAnswer distribution:")
print(sub['answer'].value_counts().sort_index().to_string())
print(f"\nTotal predictions: {len(sub)}")